# Phase 1: Exploratory Data Analysis

Explore the Telco Customer Churn dataset — understand distributions, class balance, and relationships between features and churn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('husl')

## 1. Load & Inspect

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 2. Data Cleaning

`TotalCharges` is stored as object. 11 rows contain a space string (`' '`) for customers with `tenure=0` (new customers not yet billed). Convert to numeric and fill with 0.

In [ ]:
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()
print(f'Rows with non-numeric TotalCharges: {mask.sum()}')
df.loc[mask, ['tenure', 'TotalCharges', 'Churn']]

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df = df.drop(columns='customerID')
df.describe()

## 3. Target Distribution

In [ ]:
churn_rate = df['Churn'].value_counts(normalize=True)
print(churn_rate)

sns.countplot(x='Churn', data=df)
plt.title('Distribution of Churn')
plt.show()

**Class imbalance**: ~73.5% No churn, ~26.5% Yes churn. Will address with SMOTE in modeling.

## 4. Categorical Features vs Churn

In [ ]:
cat_features = [c for c in df.columns if c not in df.describe().columns and c != 'Churn']
print(cat_features)

In [ ]:
fig, ax = plt.subplots(4, 4, figsize=(20, 20))
for idx, feature in enumerate(cat_features):
    row, col = idx // 4, idx % 4
    sns.countplot(x=feature, hue='Churn', data=df, ax=ax[row][col])
    ax[row][col].set_title(f'Churn by {feature}')
    ax[row][col].tick_params(axis='x', rotation=45)
    for container in ax[row][col].containers:
        ax[row][col].bar_label(container)
plt.tight_layout()
plt.show()

**Key observations:**
- Gender: near-equal churn rate for male/female
- Senior citizens churn more
- No partner / no dependents → higher churn
- Fiber optic internet → much higher churn than DSL
- No OnlineSecurity / TechSupport → higher churn
- Month-to-month contract → dramatically higher churn
- Electronic check payment → highest churn among payment methods

## 5. Numerical Features vs Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.histplot(x=col, hue='Churn', data=df, ax=ax)
    ax.set_title(f'{col} by Churn')
plt.tight_layout()
plt.show()

**Key observations:**
- Low tenure (0–12 months) → highest churn; drops sharply as customers stay longer
- Higher monthly charges → more churn
- Low total charges → churners (consistent with low tenure)

## 6. Boxplots: Tenure by Categorical Features

In [ ]:
groups = [
    ['gender', 'SeniorCitizen', 'Partner', 'Dependents'],
    ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity'],
    ['OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV'],
    ['StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'],
]
for group in groups:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, feature in zip(axes, group):
        sns.boxplot(x=feature, y='tenure', hue='Churn', data=df, ax=ax)
        ax.set_title(f'Tenure by {feature}')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Tenure Segmentation

In [ ]:
bins = [0, 6, 12, 18, 24, 36, 48, 60, 72, 1000]
bin_labels = ['0-6', '6-12', '12-18', '18-24', '24-36', '36-48', '48-60', '60-72', '72+']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=bin_labels, right=False)

sns.countplot(x='tenure_group', hue='Churn', data=df)
plt.title('Churn by Tenure Group')
plt.show()

## 8. Month-to-Month vs Long-Term Contract

In [ ]:
churn_num = df['Churn'].map({'Yes': 1, 'No': 0})
is_m2m = (df['Contract'] == 'Month-to-month').astype(int)
rates = churn_num.groupby(is_m2m).mean().rename(index={0: 'Long-term', 1: 'Month-to-month'})
print('Churn rate by contract type:')
print(rates)

Month-to-month customers churn at **~42.7%** vs **~6.8%** for long-term contract holders.

## 9. Correlation Heatmap

In [ ]:
df_corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_corr['Churn'] = churn_num
plt.figure(figsize=(6, 5))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## Summary of Key Findings

| Driver | Observation |
|---|---|
| Contract type | Month-to-month: 42.7% churn vs 6.8% long-term |
| Fiber optic | Much higher churn than DSL or no internet |
| Tenure | Heavy churn concentration in first 6 months |
| Monthly charges | Higher charges correlate with churn |
| Support services | No OnlineSecurity/TechSupport → higher churn |
| Demographics | Senior citizens, no partner, no dependents → higher churn |

→ Feature engineering in `02_feature_engineering.ipynb` will operationalise these patterns.